# Faqat qaror: ballsiz API'ni aldash

Oldingi labda bank API'si har ariza uchun ishonch balini qaytarardi va siz shu ball orqali modelning nimaga
qarab qaror qilishini bilib oldingiz. Bank xulosa chiqardi: endi ball yo'q. API faqat bitta so'z aytadi:
«tasdiqlandi» yoki «rad etildi».

Bir qarashda bu hujumni to'xtatadigandek. Aslida esa yo'q. Qaror ham ma'lumot, shunchaki juda oz: har so'rov
bitta «ha» yoki «yo'q» beradi. Shu «ha-yo'q»lar bilan qaror o'zgaradigan aniq nuqtani topsa bo'ladi. Bunday
hujum **decision-based** yoki **hard-label** hujum deyiladi.

Bu darsda uch narsani o'rganasiz:

1. Faqat «ha» yoki «yo'q» javob bilan chegarani qanday topishni (ikkilik qidiruv).
2. Chegaragacha masofa xususiyatning kuchi haqida nima deyishini.
3. Ballni yashirish nima uchun himoya emas, faqat hujum narxini oshirish ekanini.

Python'da ro'yxat, `for` sikli va funksiya yozishni bilsangiz yetarli. Oldingi labni («Kredit skoring») yechgan
bo'lsangiz yaxshi, lekin shart emas.

Bir gap: bu o'quv muhiti. Haqiqiy bank yoki boshqa tizimda ruxsatsiz shunday qilish jinoyat hisoblanadi.

## 1. Labga ulanamiz

Platformadagi lab sahifasini oching. U yerda **Token** degan maydon bor: undagi satrni nusxalab, pastdagi
`TOKEN` ichiga qo'ying. Token faqat sizga tegishli va bir necha soatdan keyin eskiradi. Eskirib qolsa, lab
sahifasini yangilab, yangisini oling.

In [ ]:
BASE = "https://platform.hacknow.uz/ai-poligon"
TOKEN = ""   # lab sahifasidagi Token maydonidan nusxalab, qo'shtirnoq ichiga qo'ying

assert TOKEN, "Avval TOKEN ni to'ldiring: platformadagi lab sahifasidan nusxalang."

import urllib.request

urllib.request.urlretrieve(BASE + "/client/hn_lab.py", "hn_lab.py")   # labning kichik mijozi
from hn_lab import HnLab

lab = HnLab(BASE, "ml-qaror-evasion", TOKEN)
info = lab.task()
print(info["lab"]["title"])
print("Urinish:", info["attempt"], "| sarflangan so'rov:", info["queries_used"], "/", info["query_budget"])
if info["solved"]:
    print("Bu lab sizda allaqachon yechilgan. Flag:", info.get("flag"))
if info["query_budget"] - info["queries_used"] < 70:
    print("Diqqat: so'rovlar kam qoldi. Yetmay qolsa, lab.reset() bilan yangi urinish boshlang va notebook'ni boshidan yurgizing.")

## 2. Vazifa bilan tanishamiz

Ariza oldingi labdagidek: 8 ta son, har biri 0 bilan 1 orasida. Ulardan beshtasini o'zgartirsa bo'ladi, uchtasiga
tegib bo'lmaydi.

In [ ]:
vazifa = info["task"]
nomlar = vazifa["features"]
x0 = vazifa["x0"]

for i, nom in enumerate(nomlar):
    izoh = "o'zgartirsa bo'ladi" if i in vazifa["mutable"] else "TEGIB BO'LMAYDI"
    print(i, nom, "=", x0[i], "—", izoh)
print()
print("O'zgartirish byudjeti:", vazifa["l1_budget"])

Byudjetga e'tibor bering: u ancha tor. Hamma o'zgarishlaringiz yig'indisi 0.25 dan oshmasligi kerak. Xususiyatlar
esa o'rtaroqda turibdi, ya'ni har qaysisini chetga surish uchun 0.3 dan ko'proq yo'l bosish kerak. Demak,
xususiyatni shunchaki oxirigacha surib yuborish byudjetga sig'maydi. Qaror aynan qayerda o'zgarishini topishga
to'g'ri keladi.

## 3. Birinchi so'rov: endi ball yo'q

Arizani o'zgartirmasdan yuborib, javob qanday ko'rinishda kelishiga qaraymiz.

In [ ]:
javob = lab.score({"rows": [x0]})
print(javob["output"][0])
print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

Javobda faqat ikki narsa bor: `decision` — qaror, `cheklov` — ariza qoidalarga to'g'ri keladimi. Ball yo'q.

So'rovlar ham cheklangan: bitta urinishda 200 ta qator yubora olasiz. So'rovlar tugab qolsa, `lab.reset()` yangi
urinish boshlaydi, lekin unda ariza ham, model ham yangilanadi.

## 4. Xususiyatni oxirigacha surib ko'ramiz

Daromadni bir marta eng past qiymatga (0.0), bir marta eng yuqori qiymatga (1.0) qo'yib ko'ramiz.

In [ ]:
past_chet, yuqori_chet = list(x0), list(x0)
past_chet[0] = 0.0
yuqori_chet[0] = 1.0

javob = lab.score({"rows": [past_chet, yuqori_chet]})
print("daromad = 0.0:", javob["output"][0])
print("daromad = 1.0:", javob["output"][1])

Ikki narsaga e'tibor bering.

Birinchisi, `cheklov` ustunida `l1_byudjet` chiqdi: bunday katta o'zgarish byudjetga sig'maydi. Lekin model
baribir qarorini aytdi. Demak, sinov arizalari qoidaga to'g'ri kelishi shart emas, faqat yakuniy ariza byudjet
ichida bo'lishi kerak.

Ikkinchisi, qaror o'zgargan bo'lishi ham, o'zgarmagan bo'lishi ham mumkin. O'zgargan bo'lsa, bu juda qimmatli
xabar: asl qiymat bilan chet qiymat orasida qayerdadir **chegara** bor. Chegaradan bir tomonda ariza rad
etiladi, ikkinchi tomonda tasdiqlanadi. Bizga aynan o'sha nuqta kerak.

## 5. Ikkilik qidiruv

Do'stingiz 1 dan 1000 gacha bo'lgan son o'yladi, siz esa faqat «shu sondan kattami?» deb so'ray olasiz. Eng
tez yo'l — har safar oraliqning o'rtasini so'rash. «500 dan kattami?» — yo'q. Demak, son 1 bilan 500 orasida.
«250 dan kattami?» — ha. Demak, 250 bilan 500 orasida. Har savol oraliqni ikki baravar qisqartiradi va o'n
savoldan keyin bitta son qoladi.

Chegarani ham xuddi shunday topamiz. Ikki masofani bilamiz: `past` — shuncha surilganda ariza hali rad etiladi,
`yuqori` — shuncha surilganda tasdiqlanadi. O'rtasini sinab ko'ramiz va javobga qarab oraliqning yarmini
tashlab yuboramiz.

Pastdagi katak buni serverga so'rov yubormasdan ko'rsatadi: chegara ataylab 0.137 ga qo'yilgan.

In [ ]:
yashirin_chegara = 0.137
past, yuqori = 0.0, 0.5

for qadam in range(1, 6):
    orta = (past + yuqori) / 2
    tasdiq = orta >= yashirin_chegara
    print(qadam, "- qadam: o'rta =", round(orta, 4), "->", "tasdiqlanadi" if tasdiq else "rad etiladi")
    if tasdiq:
        yuqori = orta
    else:
        past = orta

print("5 qadamdan keyin chegara shu oraliqda:", round(past, 4), "...", round(yuqori, 4))

Besh qadamda 0.5 kenglikdagi oraliq 0.016 gacha qisqardi. O'n qadamda u 0.0005 bo'ladi: bizga shuncha aniqlik
yetib ortadi.

## 6. Topshiriq 1: `chegara_top` funksiyasini yozing

Yuqoridagi siklni funksiyaga aylantiring. Funksiya `tasdiqlanadimi` degan boshqa funksiyani qabul qiladi: unga
masofa berilsa, `True` (tasdiqlanadi) yoki `False` (rad etiladi) qaytaradi. Oxirida funksiya `yuqori` ni
qaytarsin: bu tasdiqlanishi aniq bo'lgan eng kichik masofa.

In [ ]:
def chegara_top(tasdiqlanadimi, past, yuqori, qadamlar=10):
    """`past` masofada ariza hali rad etiladi, `yuqori` masofada tasdiqlanadi. Tasdiqlanadigan eng kichik masofani qaytaradi."""
    # TOPSHIRIQ 1: 5-bo'limdagi kabi sikl yozing, u `qadamlar` marta aylansin. Har aylanishda o'rtani hisoblang,
    # tasdiqlanadimi(orta) ni chaqiring va javobga qarab `yuqori` yoki `past` ni o'rtaga tenglang.
    return yuqori

Keyingi katak funksiyangizni serverga so'rov yubormasdan tekshiradi: chegarasi oldindan ma'lum bo'lgan soxta
«model»da sinaydi va funksiya necha marta so'raganini sanaydi.

In [ ]:
chaqiruvlar = []


def soxta_model(masofa):
    chaqiruvlar.append(masofa)
    return masofa >= 0.137


topilgan = chegara_top(soxta_model, 0.0, 0.5)
funksiya_tayyor = 0.137 <= topilgan <= 0.138 and len(chaqiruvlar) == 10
if funksiya_tayyor:
    print("Funksiya to'g'ri ishlayapti: chegara", round(topilgan, 4), "deb topildi, 10 ta so'rov bilan.")
elif not chaqiruvlar:
    print("Hali tayyor emas: funksiya modeldan hech narsa so'ramadi. Topshiriq 1 ni bajaring.")
else:
    print("Hali tayyor emas: funksiya", round(topilgan, 4), "qaytardi va", len(chaqiruvlar), "marta so'radi.")
    print("Kutilgan: 0.137 atrofida, aynan 10 ta so'rov bilan.")

## 7. Qaysi xususiyatlar qarorni o'zgartiradi?

Endi haqiqiy modelga qaytamiz. Avval har bir o'zgaruvchan xususiyatni ikkala chetga surib ko'ramiz: qaysi
biri qarorni o'zgartira oladi? Bu o'nta ariza, hammasini bitta chaqiruvda yuboramiz.

Natija `ochiladigan` ro'yxatiga tushadi. Har element uchta narsadan iborat: xususiyat raqami, yo'nalish
(`+1` — oshirish, `-1` — kamaytirish) va chetgacha bo'lgan masofa.

In [ ]:
sinovlar = []
for i in vazifa["mutable"]:
    for chet in (0.0, 1.0):
        ariza = list(x0)
        ariza[i] = chet
        sinovlar.append((i, chet, ariza))

javob = lab.score({"rows": [ariza for _, _, ariza in sinovlar]})

ochiladigan = []   # har element: (xususiyat raqami, yo'nalish, chetgacha masofa)
for (i, chet, _), natija in zip(sinovlar, javob["output"]):
    if natija["decision"] == "tasdiqlandi":
        yonalish = 1 if chet > x0[i] else -1
        ochiladigan.append((i, yonalish, abs(chet - x0[i])))
        print(i, nomlar[i], "->", chet, ": qaror o'zgaradi, chetgacha masofa", round(abs(chet - x0[i]), 3))

print("Qarorni o'zgartira oladigan yo'llar:", len(ochiladigan))
print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])

Ro'yxatga tushmagan xususiyatlar bu modelda juda kuchsiz: ularni oxirigacha sursangiz ham qaror o'zgarmaydi.

Ro'yxatdagilarning har birida `x0` bilan chet orasida chegara bor. Chegara qanchalik yaqin bo'lsa, o'sha
xususiyat shunchalik kuchli: ozgina surish qarorni o'zgartirishga yetadi. Bizga eng yaqin chegara kerak, chunki
faqat u byudjetga sig'adi.

## 8. Topshiriq 2: har bir yo'l uchun chegarani toping

Pastdagi `sorov_yasa` tayyor. U xususiyat raqami va yo'nalishni olib, `chegara_top` ga beriladigan funksiyani
yasab beradi: o'sha funksiya arizani kerakli masofaga surib, serverdan qarorni so'raydi.

In [ ]:
def sorov_yasa(i, yonalish):
    """i-xususiyatni x0 dan berilgan masofaga suradigan va server tasdiqladimi-yo'qmi qaytaradigan funksiya."""
    def tasdiqlanadimi(masofa):
        ariza = list(x0)
        ariza[i] = x0[i] + yonalish * masofa
        return lab.score({"rows": [ariza]})["output"][0]["decision"] == "tasdiqlandi"
    return tasdiqlanadimi

Endi `ochiladigan` ro'yxatidagi har bir yo'l uchun `chegara_top` ni chaqiring. Asl ariza rad etilgan, demak
`past` 0.0 dan boshlanadi. Chetda ariza tasdiqlanadi, demak `yuqori` chetgacha masofaga teng. Natijani
`natijalar` ro'yxatiga uchlik qilib qo'shing: `(chegaragacha masofa, xususiyat raqami, yo'nalish)`.

Har chaqiruv o'nta so'rov sarflaydi, shuning uchun bu katakni keraksiz qayta-qayta ishga tushirmang. So'rovlar
tugab qolsa `lab.reset()` yangi urinish boshlaydi; shundan keyin notebook'ni boshidan yurgizing.

In [ ]:
natijalar = []   # har element: (chegaragacha masofa, xususiyat raqami, yo'nalish)

# TOPSHIRIQ 2: siklni shu yerga yozing.

Keyingi katak eng yaqin chegarani tanlaydi, yakuniy arizani tuzadi, uni avval o'zi tekshiradi (bu so'rov
sarflamaydi) va shundan keyingina yuboradi.

In [ ]:
def tekshir(ariza):
    if len(ariza) != len(x0):
        print("Xato: arizada", len(x0), "ta son bo'lishi kerak.")
        return False
    sarf = sum(abs(a - b) for a, b in zip(ariza, x0))
    print("Sarflangan byudjet: %.3f / %.2f" % (sarf, vazifa["l1_budget"]))
    if any(ariza[i] != x0[i] for i in vazifa["immutable"]):
        print("Xato: tegib bo'lmaydigan xususiyat o'zgargan.")
        return False
    if any(not 0.0 <= qiymat <= 1.0 for qiymat in ariza):
        print("Xato: hamma qiymat 0 bilan 1 orasida bo'lishi kerak.")
        return False
    if sarf > vazifa["l1_budget"] + 1e-9:
        print("Xato: byudjetdan oshib ketdi. Chegara to'g'ri topildimi?")
        return False
    return True


if not funksiya_tayyor:
    print("Hali tayyor emas: avval Topshiriq 1 dagi funksiyani yozing.")
elif len(natijalar) != len(ochiladigan):
    print("Hali tayyor emas: natijalar ichida", len(natijalar), "ta element bor, kerak:", len(ochiladigan), "(Topshiriq 2).")
else:
    natijalar.sort()
    for masofa, i, yonalish in natijalar:
        print("%d %-22s %s tomonga: chegaragacha %.3f" % (i, nomlar[i], "oshirish" if yonalish > 0 else "kamaytirish", masofa))
    masofa, i, yonalish = natijalar[0]
    yakuniy = list(x0)
    yakuniy[i] = x0[i] + yonalish * masofa
    if tekshir(yakuniy):
        javob = lab.score({"rows": [yakuniy]})
        print(javob["output"][0])
        print("Sarflangan so'rov:", javob["queries_used"], "/", javob["query_budget"])
        if javob["solved"]:
            print("Bo'ldi! Model arizani tasdiqladi.")
            print("Flag:", javob["flag"])
        else:
            print("Hali tasdiqlanmadi. Yakuniy ariza chegaradan o'tganini tekshiring.")

Flag chiqqan bo'lsa, uni platformadagi topshiriq sahifasiga yuboring. Flag faqat sizniki: boshqa odamning
flag'i sizda ishlamaydi.

## 9. Nima bo'ldi?

Bank ballni yashirdi, lekin siz baribir uddaladingiz. Farqi faqat narxda: ball bor paytda o'n beshtacha so'rov
yetgan edi, endi bir necha barobar ko'p ketdi. Ballni yashirish hujumni to'xtatmaydi, uni qimmatroq qiladi, xolos.

Sababi oddiy. Model qaror chiqarar ekan, qaror qayerdadir o'zgarishi kerak. O'sha joyni topish uchun esa
«ha» va «yo'q»ning o'zi kifoya. Har so'rov noaniqlikni ikki baravar kamaytiradi, shuning uchun aniqlik juda
tez o'sadi.

Himoyachi nima qilishi mumkin? Eng foydalisi — so'rovlarni kuzatish. Sizning so'rovlaringiz oddiy mijoznikiga
umuman o'xshamadi: bitta odamning o'nlab arizasi keldi, ular bir-biridan faqat bitta sonning mayda qismi bilan
farq qildi va tobora bitta nuqtaga yaqinlashib bordi. Bunday izni payqash mumkin. Ikkinchi yo'l — so'rovlar
sonini cheklash: ikkilik qidiruvga har chegara uchun o'nta so'rov kerak, cheklov qattiq bo'lsa hujum sekinlashadi.
Uchinchisi — arizadagi ma'lumotni hujjat bilan tasdiqlash: sonni qog'ozda o'zgartirish oson, hayotda qiyin.

## Qo'shimcha (xohlasangiz)

1. Ikki xususiyatni birga o'zgartirsangiz, jami o'zgarish kamayadimi? Avval taxmin qiling, keyin sinab ko'ring.
2. Bitta chaqiruvda 20 tagacha ariza yuborish mumkin. Chegarani o'nta emas, ikki-uchta chaqiruv bilan topsa
   bo'ladimi? Qanday?
3. Bank chegaraga yaqin arizalarga ataylab tasodifiy javob bera boshlasa, qidiruvingiz qanday buziladi va uni
   qanday tuzatgan bo'lardingiz?